
# Notebook 10 — Executive Summary & Final Validation

## End-to-End Hardware Supply Network Optimizer

### Objective
Consolidate and validate the results of a 12-week hardware supply network optimization study, integrating demand forecasting, supplier sourcing, manufacturing, distribution, transportation, inventory planning, and disruption resilience.

### Business Questions
1. How much does optimization reduce total network costs?
2. Which sourcing, manufacturing, and transportation decisions generate the savings?
3. How resilient is the network under supply and demand disruptions?
4. Which capacity expansions deliver the greatest modeled financial benefit?
5. What operational recommendations follow from the results?

### Methodology
- Demand inputs integrated from Project 1.
- Hypothetical hardware supply network and operating assumptions.
- Rule-based baseline versus Pyomo/HiGHS optimization.
- Supplier, plant, transportation, and inventory constraints.
- Reoptimized disruption and sensitivity experiments.
- Four executive Tableau dashboards.

### Deliverables
- Validated cost and operational KPI summaries.
- Scenario and sensitivity findings.
- Dashboard deliverable audit.
- Executive recommendations and limitations.
- Final V1 executive summary and supporting report.

**Note:** Financial outcomes represent modeled results from hypothetical assumptions, not realized company savings.

In [1]:
# STEP 01 — Initialize Project 2 environment

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Locate the project root automatically
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "notebooks").is_dir()
        and (path / "outputs").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the Project 2 root folder."
    )

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLEAU_DIR = PROJECT_ROOT / "outputs" / "tableau"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
DOCS_DIR = PROJECT_ROOT / "docs"

DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("NOTEBOOK 10 — EXECUTIVE SUMMARY & FINAL VALIDATION")
print("=" * 65)

print(f"Project root: {PROJECT_ROOT}")
print(f"Tables folder: {TABLES_DIR}")
print(f"Tableau folder: {TABLEAU_DIR}")
print(f"Figures folder: {FIGURES_DIR}")
print(f"Docs folder: {DOCS_DIR}")

assert TABLES_DIR.exists(), "Outputs/tables folder missing."

print("\nSTEP 01: PASSED")


NOTEBOOK 10 — EXECUTIVE SUMMARY & FINAL VALIDATION
Project root: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer
Tables folder: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables
Tableau folder: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tableau
Figures folder: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/figures
Docs folder: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/docs

STEP 01: PASSED


In [2]:
# STEP 02 — Audit required project outputs

required_tables = [
    "baseline_vs_optimized.csv",
    "optimization_savings_drivers.csv",
    "scenario_comparison.csv",
    "scenario_cost_breakdown.csv",
    "supplier_allocation_comparison.csv",
    "manufacturing_capacity_analysis.csv",
    "distribution_performance_kpis.csv",
]

expected_dashboards = [
    "dashboard_01_network_cost.pdf",
    "dashboard_02_sourcing_manufacturing.pdf",
    "dashboard_03_transportation_distribution.pdf",
    "dashboard_04_risk_scenario_sensitivity.pdf",
]

audit_rows = []

for name in required_tables:
    path = TABLES_DIR / name
    audit_rows.append({
        "Category": "Analysis",
        "File": name,
        "Found": path.exists()
    })

for name in expected_dashboards:
    path = FIGURES_DIR / name
    audit_rows.append({
        "Category": "Dashboard",
        "File": name,
        "Found": path.exists()
    })

audit_df = pd.DataFrame(audit_rows)

display(audit_df)

print("\nFiles found:", int(audit_df["Found"].sum()))
print("Files checked:", len(audit_df))
print("Files needing review:", int((~audit_df["Found"]).sum()))

print("\nSTEP 02: AUDIT COMPLETE")


,Category,File,Found
0,Analysis,baseline_vs_optimized.csv,True
1,Analysis,optimization_savings_drivers.csv,True
2,Analysis,scenario_comparison.csv,True
3,Analysis,scenario_cost_breakdown.csv,True
4,Analysis,supplier_allocation_comparison.csv,True
5,Analysis,manufacturing_capacity_analysis.csv,True
6,Analysis,distribution_performance_kpis.csv,True
7,Dashboard,dashboard_01_network_cost.pdf,True
8,Dashboard,dashboard_02_sourcing_manufacturing.pdf,True
9,Dashboard,dashboard_03_transportation_distribution.pdf,True



Files found: 11
Files checked: 11
Files needing review: 0

STEP 02: AUDIT COMPLETE


In [3]:
# STEP 03 — Inspect core financial results

core_files = {
    "Baseline vs Optimized":
        TABLES_DIR / "baseline_vs_optimized.csv",

    "Disruption Scenarios":
        TABLES_DIR / "scenario_comparison.csv",

    "Demand Sensitivity":
        TABLEAU_DIR / "04_risk" / "demand_sensitivity.csv",
}

core_results = {}

for label, path in core_files.items():

    print("\n" + "=" * 65)
    print(label.upper())
    print("=" * 65)

    if not path.exists():
        print(f"FILE NOT FOUND: {path}")
        continue

    df = pd.read_csv(path)
    core_results[label] = df

    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print(f"Column names: {list(df.columns)}")

    display(df.head(10))

print("\nSTEP 03: CORE RESULTS INSPECTED")



BASELINE VS OPTIMIZED
Rows: 4
Columns: 2
Column names: ['Metric', 'Value']


,Metric,Value
0,Baseline Cost,414139.700000
1,Optimized Cost,404131.200000
2,Cost Savings,10008.500000
3,Cost Improvement (%),2.416697



DISRUPTION SCENARIOS
Rows: 5
Columns: 6
Column names: ['Scenario', 'Status', 'Total_Demand_Units', 'Optimized_Cost_USD', 'Incremental_Cost_USD', 'Cost_Change_Pct']


,Scenario,Status,Total_Demand_Units,Optimized_Cost_USD,Incremental_Cost_USD,Cost_Change_Pct
0,Reference,Optimal,6180,404131.20,0.00,0.000000
1,Supplier_Disruption,Optimal,6180,417631.20,13500.00,3.340499
2,Demand_Surge,Optimal,6336,418967.70,14836.50,3.671209
3,Manufacturing_Disruption,Optimal,6180,405741.20,1610.00,0.398385
4,Freight_Inflation,Optimal,6180,413920.45,9789.25,2.422295



DEMAND SENSITIVITY
Rows: 4
Columns: 8
Column names: ['Experiment', 'Demand_Growth_Pct', 'Total_Demand_Units', 'Additional_Demand_Units', 'Status', 'Optimized_Cost_USD', 'Incremental_Cost_USD', 'Incremental_Cost_Per_Added_Unit_USD']


,Experiment,Demand_Growth_Pct,Total_Demand_Units,Additional_Demand_Units,Status,Optimized_Cost_USD,Incremental_Cost_USD,Incremental_Cost_Per_Added_Unit_USD
0,Demand_Growth_Plus_5Pct,5,6480,300,Optimal,433341.45,29210.25,97.367500
1,Demand_Growth_Plus_10Pct,10,6804,624,Optimal,467764.55,63633.35,101.976522
2,Demand_Growth_Plus_15Pct,15,7104,924,infeasible,NaN,NaN,NaN
3,Demand_Growth_Plus_20Pct,20,7416,1236,infeasible,NaN,NaN,NaN



STEP 03: CORE RESULTS INSPECTED


In [4]:
# STEP 04 — Financial and demand sensitivity validation

# Load the previously inspected datasets
financial = core_results["Baseline vs Optimized"].copy()
scenarios = core_results["Disruption Scenarios"].copy()
demand = core_results["Demand Sensitivity"].copy()

# 1. Validate baseline and optimized network costs
metrics = financial.set_index("Metric")["Value"].astype(float)

baseline_cost = float(metrics["Baseline Cost"])
optimized_cost = float(metrics["Optimized Cost"])
reported_savings = float(metrics["Cost Savings"])

calculated_savings = baseline_cost - optimized_cost
savings_pct = calculated_savings / baseline_cost * 100

assert np.isclose(calculated_savings, reported_savings)
assert np.isclose(baseline_cost, 414139.70)
assert np.isclose(optimized_cost, 404131.20)

print("FINANCIAL VALIDATION")
print(f"Baseline cost:     ${baseline_cost:,.2f}")
print(f"Optimized cost:    ${optimized_cost:,.2f}")
print(f"Modeled savings:   ${calculated_savings:,.2f}")
print(f"Improvement:       {savings_pct:.2f}%")

# 2. Validate disruption cost calculations
scenarios["Recomputed_Incremental_USD"] = (
    scenarios["Optimized_Cost_USD"] - optimized_cost
)

assert np.allclose(
    scenarios["Recomputed_Incremental_USD"],
    scenarios["Incremental_Cost_USD"],
    atol=0.02
)

print("\nDISRUPTION SCENARIOS: VALIDATED")

display(
    scenarios[
        ["Scenario", "Status",
         "Optimized_Cost_USD",
         "Incremental_Cost_USD"]
    ]
)

# 3. Validate demand growth scenarios
demand["Demand_Growth_Pct"] = pd.to_numeric(
    demand["Demand_Growth_Pct"]
)

demand["Recomputed_Incremental_USD"] = (
    demand["Optimized_Cost_USD"] - optimized_cost
)

feasible = demand[
    demand["Status"].str.lower() == "optimal"
].copy()

infeasible = demand[
    demand["Status"].str.lower() == "infeasible"
].copy()

assert len(feasible) == 2
assert len(infeasible) == 2

assert set(feasible["Demand_Growth_Pct"]) == {5, 10}
assert set(infeasible["Demand_Growth_Pct"]) == {15, 20}

if "Incremental_Cost_USD" in feasible.columns:
    assert np.allclose(
        feasible["Incremental_Cost_USD"],
        feasible["Recomputed_Incremental_USD"],
        atol=0.02
    )

# 4. Explain the prior Tableau aggregation error
growth_sum = demand["Demand_Growth_Pct"].sum()
incremental_sum = feasible[
    "Recomputed_Incremental_USD"
].sum()

print("\nDEMAND SENSITIVITY: VALIDATED")

display(
    demand[
        ["Experiment", "Demand_Growth_Pct",
         "Status", "Optimized_Cost_USD",
         "Recomputed_Incremental_USD"]
    ]
)

print(f"\nSum of growth percentages: {growth_sum:.0f}%")
print(f"Sum of feasible cost increases: ${incremental_sum:,.2f}")
print("These sums are NOT a single demand-growth scenario.")

print("\nHighest tested feasible growth: +10%")
print("Tested infeasible growth levels: +15%, +20%")

print("\nSTEP 04: VALIDATION PASSED")


FINANCIAL VALIDATION
Baseline cost:     $414,139.70
Optimized cost:    $404,131.20
Modeled savings:   $10,008.50
Improvement:       2.42%

DISRUPTION SCENARIOS: VALIDATED


,Scenario,Status,Optimized_Cost_USD,Incremental_Cost_USD
0,Reference,Optimal,404131.20,0.00
1,Supplier_Disruption,Optimal,417631.20,13500.00
2,Demand_Surge,Optimal,418967.70,14836.50
3,Manufacturing_Disruption,Optimal,405741.20,1610.00
4,Freight_Inflation,Optimal,413920.45,9789.25



DEMAND SENSITIVITY: VALIDATED


,Experiment,Demand_Growth_Pct,Status,Optimized_Cost_USD,Recomputed_Incremental_USD
0,Demand_Growth_Plus_5Pct,5,Optimal,433341.45,29210.25
1,Demand_Growth_Plus_10Pct,10,Optimal,467764.55,63633.35
2,Demand_Growth_Plus_15Pct,15,infeasible,NaN,NaN
3,Demand_Growth_Plus_20Pct,20,infeasible,NaN,NaN



Sum of growth percentages: 50%
Sum of feasible cost increases: $92,843.60
These sums are NOT a single demand-growth scenario.

Highest tested feasible growth: +10%
Tested infeasible growth levels: +15%, +20%

STEP 04: VALIDATION PASSED


In [6]:
# STEP 05 — SOURCING, MANUFACTURING & DISTRIBUTION VALIDATION

import re
import pandas as pd
import numpy as np
from IPython.display import display


# ------------------------------------------------------------
# 5.1 — Helper: Load and standardize Tableau-ready CSVs
# ------------------------------------------------------------

def load_tableau_csv(folder, filename):

    path = TABLEAU_DIR / folder / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)

    df.columns = [
        re.sub(r"[^a-zA-Z0-9]+", "_", str(col))
        .strip("_")
        .lower()
        for col in df.columns
    ]

    print(f"Loaded {filename}: {len(df):,} rows")

    return df


# ------------------------------------------------------------
# 5.2 — Helper: Select the optimized/reference scenario
# ------------------------------------------------------------

def optimized_rows(df):

    # Datasets comparing baseline and optimized plans
    if "plan" in df.columns:

        labels = df["plan"].astype(str).str.strip().str.lower()

        if "optimized" in labels.values:
            return df.loc[labels == "optimized"].copy()

        if "reference" in labels.values:
            return df.loc[labels == "reference"].copy()

        raise ValueError(
            f"Cannot identify optimized plan. "
            f"Available values: {sorted(labels.unique())}"
        )

    # Scenario datasets generally call the optimized model
    # "Reference" because it is the reference for stress tests.
    if "scenario" in df.columns:

        labels = df["scenario"].astype(str).str.strip().str.lower()

        if "reference" in labels.values:
            return df.loc[labels == "reference"].copy()

        if "optimized" in labels.values:
            return df.loc[labels == "optimized"].copy()

        # Accept a single unambiguous scenario label.
        if labels.nunique() == 1:
            print(
                f"INFO: Using sole scenario label: "
                f"{labels.iloc[0]}"
            )
            return df.copy()

        raise ValueError(
            f"Cannot identify optimized scenario. "
            f"Available values: {sorted(labels.unique())}"
        )

    # Some reporting files contain only the optimized solution.
    print("INFO: No Plan/Scenario column; using all records.")
    return df.copy()


# ------------------------------------------------------------
# 5.3 — Load Notebook 09 reporting datasets
# ------------------------------------------------------------

supplier = load_tableau_csv(
    "02_sourcing",
    "supplier_weekly.csv"
)

manufacturing = load_tableau_csv(
    "02_sourcing",
    "manufacturing_weekly.csv"
)

freight = load_tableau_csv(
    "03_distribution",
    "freight_modes.csv"
)

dc = load_tableau_csv(
    "03_distribution",
    "dc_weekly.csv"
)


# ------------------------------------------------------------
# 5.4 — Supplier allocation and sourcing concentration
# ------------------------------------------------------------

supplier_opt = optimized_rows(supplier)

supplier_opt["order_units"] = pd.to_numeric(
    supplier_opt["order_units"]
)

supplier_allocation = (
    supplier_opt
    .groupby("supplier_id")["order_units"]
    .sum()
    .sort_values(ascending=False)
)

total_order_units = supplier_allocation.sum()

assert total_order_units > 0, "No supplier purchase orders found."
assert "SUP_A" in supplier_allocation.index

supplier_a_units = supplier_allocation.loc["SUP_A"]

supplier_a_share = (
    supplier_a_units / total_order_units * 100
)

print("\n" + "=" * 65)
print("SUPPLIER ALLOCATION")
print("=" * 65)

display(
    supplier_allocation.to_frame("Order Units")
)

print(f"Total component orders: {total_order_units:,.0f}")
print(f"Supplier A orders: {supplier_a_units:,.0f}")
print(f"Supplier A sourcing share: {supplier_a_share:.1f}%")


# ------------------------------------------------------------
# 5.5 — Manufacturing production and capacity utilization
# ------------------------------------------------------------

manufacturing_opt = optimized_rows(manufacturing)

manufacturing_opt["production_units"] = pd.to_numeric(
    manufacturing_opt["production_units"]
)

manufacturing_opt["available_capacity"] = pd.to_numeric(
    manufacturing_opt["available_capacity"]
)

weekly_production = (
    manufacturing_opt
    .groupby("week", as_index=False)
    .agg(
        production_units=("production_units", "sum"),
        available_capacity=("available_capacity", "max")
    )
    .sort_values("week")
)

total_production = weekly_production["production_units"].sum()

weeks_at_capacity = int(
    (
        weekly_production["production_units"]
        >= weekly_production["available_capacity"] - 0.01
    ).sum()
)

peak_production = weekly_production["production_units"].max()

weekly_production["utilization_pct"] = (
    weekly_production["production_units"]
    / weekly_production["available_capacity"]
    * 100
)

print("\n" + "=" * 65)
print("MANUFACTURING PERFORMANCE")
print("=" * 65)

print(f"Total production: {total_production:,.0f} units")
print(f"Peak weekly production: {peak_production:,.0f} units")
print(f"Weeks at capacity: {weeks_at_capacity}")

display(weekly_production)


# ------------------------------------------------------------
# 5.6 — Transportation cost performance
# ------------------------------------------------------------

freight["freight_cost_usd"] = pd.to_numeric(
    freight["freight_cost_usd"]
)

freight_by_plan = (
    freight
    .groupby("plan")["freight_cost_usd"]
    .sum()
)

baseline_freight = float(
    freight_by_plan.loc["Baseline"]
)

optimized_freight = float(
    freight_by_plan.loc["Optimized"]
)

freight_savings = baseline_freight - optimized_freight

freight_savings_pct = (
    freight_savings / baseline_freight * 100
)

print("\n" + "=" * 65)
print("TRANSPORTATION PERFORMANCE")
print("=" * 65)

print(f"Baseline freight cost: ${baseline_freight:,.2f}")
print(f"Optimized freight cost: ${optimized_freight:,.2f}")
print(f"Transportation savings: ${freight_savings:,.2f}")
print(f"Freight cost reduction: {freight_savings_pct:.2f}%")


# ------------------------------------------------------------
# 5.7 — Distribution center fulfillment
# ------------------------------------------------------------

dc_opt = optimized_rows(dc)

dc_opt["customer_demand"] = pd.to_numeric(
    dc_opt["customer_demand"]
)

dc_opt["fulfilled_units"] = pd.to_numeric(
    dc_opt["fulfilled_units"]
)

demand_units = dc_opt["customer_demand"].sum()

fulfilled_units = dc_opt["fulfilled_units"].sum()

assert demand_units > 0, "Customer demand must be positive."

service_level = (
    fulfilled_units / demand_units * 100
)

print("\n" + "=" * 65)
print("DISTRIBUTION PERFORMANCE")
print("=" * 65)

print(f"Total customer demand: {demand_units:,.0f} units")
print(f"Fulfilled demand: {fulfilled_units:,.0f} units")
print(f"Customer service level: {service_level:.1f}%")


# ------------------------------------------------------------
# 5.8 — Independent operational validations
# ------------------------------------------------------------

assert np.isclose(
    total_production, 5665
), "Production total does not match optimized model."

assert np.isclose(
    freight_savings, 8900.00, atol=0.05
), "Transportation savings do not reconcile."

assert np.isclose(
    demand_units, 6180
), "Customer demand does not match reference model."

assert np.isclose(
    service_level, 100.0
), "Customer fulfillment is below expected level."

assert (
    weekly_production["production_units"]
    <= weekly_production["available_capacity"] + 0.01
).all(), "Manufacturing capacity constraint violated."

print("\nALL OPERATIONAL VALIDATIONS PASSED")


# ------------------------------------------------------------
# 5.9 — Create executive operational KPI summary
# ------------------------------------------------------------

operational_kpis = pd.DataFrame([
    {
        "Metric": "Supplier A Share (%)",
        "Value": supplier_a_share
    },
    {
        "Metric": "Total Production Units",
        "Value": total_production
    },
    {
        "Metric": "Peak Weekly Production Units",
        "Value": peak_production
    },
    {
        "Metric": "Weeks at Plant Capacity",
        "Value": weeks_at_capacity
    },
    {
        "Metric": "Baseline Freight Cost USD",
        "Value": baseline_freight
    },
    {
        "Metric": "Optimized Freight Cost USD",
        "Value": optimized_freight
    },
    {
        "Metric": "Freight Savings USD",
        "Value": freight_savings
    },
    {
        "Metric": "Freight Reduction (%)",
        "Value": freight_savings_pct
    },
    {
        "Metric": "Customer Demand Units",
        "Value": demand_units
    },
    {
        "Metric": "Fulfilled Demand Units",
        "Value": fulfilled_units
    },
    {
        "Metric": "Customer Fulfillment (%)",
        "Value": service_level
    }
])

output_path = TABLES_DIR / "executive_operational_kpis.csv"

operational_kpis.to_csv(
    output_path,
    index=False
)

print("\nEXECUTIVE OPERATIONAL KPI SUMMARY")
display(operational_kpis)

assert output_path.exists()

print(f"\nSaved: {output_path}")
print("\nSTEP 05: OPERATIONAL VALIDATION PASSED")


Loaded supplier_weekly.csv: 72 rows
Loaded manufacturing_weekly.csv: 12 rows
Loaded freight_modes.csv: 8 rows
Loaded dc_weekly.csv: 48 rows

SUPPLIER ALLOCATION


,Order Units
supplier_id,
SUP_A,12252
SUP_B,1443


Total component orders: 13,695
Supplier A orders: 12,252
Supplier A sourcing share: 89.5%

MANUFACTURING PERFORMANCE
Total production: 5,665 units
Peak weekly production: 580 units
Weeks at capacity: 3


,week,production_units,available_capacity,utilization_pct
0,1,580,580,100.000000
1,2,580,580,100.000000
2,3,546,580,94.137931
3,4,580,580,100.000000
4,5,450,580,77.586207
5,6,570,580,98.275862
6,7,525,580,90.517241
7,8,450,580,77.586207
8,9,450,580,77.586207
9,10,450,580,77.586207



TRANSPORTATION PERFORMANCE
Baseline freight cost: $34,556.50
Optimized freight cost: $25,656.50
Transportation savings: $8,900.00
Freight cost reduction: 25.75%

DISTRIBUTION PERFORMANCE
Total customer demand: 6,180 units
Fulfilled demand: 6,180 units
Customer service level: 100.0%

ALL OPERATIONAL VALIDATIONS PASSED

EXECUTIVE OPERATIONAL KPI SUMMARY


,Metric,Value
0,Supplier A Share (%),89.463308
1,Total Production Units,5665.000000
2,Peak Weekly Production Units,580.000000
3,Weeks at Plant Capacity,3.000000
4,Baseline Freight Cost USD,34556.500000
5,Optimized Freight Cost USD,25656.500000
6,Freight Savings USD,8900.000000
7,Freight Reduction (%),25.754923
8,Customer Demand Units,6180.000000
9,Fulfilled Demand Units,6180.000000



Saved: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables/executive_operational_kpis.csv

STEP 05: OPERATIONAL VALIDATION PASSED


In [7]:
# STEP 06 — Consolidate risk and capacity-investment findings

# 1. Load capacity investment experiments
capacity = load_tableau_csv(
    "04_risk", "capacity_value.csv"
)

# 2. Select the best tested capacity experiment per lever
# These experiments are alternatives, NOT additive savings.

capacity_valid = capacity.dropna(
    subset=["lever", "total_savings_usd"]
).copy()

best_capacity = (
    capacity_valid
    .sort_values("total_savings_usd", ascending=False)
    .drop_duplicates(subset=["lever"])
    .reset_index(drop=True)
)

capacity_columns = [
    "lever",
    "extra_weekly_capacity",
    "total_savings_usd",
    "marginal_value_per_unit_week_usd",
    "maximum_12_week_investment_usd"
]

print("\nBEST TESTED CAPACITY EXPANSIONS")

display(best_capacity[
    [c for c in capacity_columns if c in best_capacity.columns]
])


# 3. Rank disruption scenarios by incremental cost
risk = core_results["Disruption Scenarios"].copy()

risk = risk[
    risk["Scenario"].astype(str).str.lower() != "reference"
].copy()

risk = risk.sort_values(
    "Incremental_Cost_USD",
    ascending=False
)

print("\nDISRUPTION COST RANKING")

display(risk[
    ["Scenario", "Optimized_Cost_USD",
     "Incremental_Cost_USD", "Cost_Change_Pct"]
])


# 4. Summarize demand feasibility
demand = core_results["Demand Sensitivity"].copy()

feasible = demand[
    demand["Status"].str.lower() == "optimal"
].copy()

infeasible = demand[
    demand["Status"].str.lower() == "infeasible"
].copy()

max_feasible_growth = feasible["Demand_Growth_Pct"].max()
first_infeasible_growth = infeasible["Demand_Growth_Pct"].min()

print("\nDEMAND GROWTH FEASIBILITY")
print(f"Highest tested feasible growth: +{max_feasible_growth:.0f}%")
print(f"First tested infeasible growth: +{first_infeasible_growth:.0f}%")

display(demand[
    ["Experiment", "Demand_Growth_Pct",
     "Status", "Optimized_Cost_USD"]
])


# 5. Extract executive findings
highest_risk = risk.iloc[0]
best_investment = best_capacity.iloc[0]

print("\nEXECUTIVE FINDINGS")

print(
    f"Highest scenario cost increase: "
    f"{highest_risk['Scenario']} "
    f"(${highest_risk['Incremental_Cost_USD']:,.2f})"
)

print(
    f"Highest tested capacity savings: "
    f"{best_investment['lever']} "
    f"(${best_investment['total_savings_usd']:,.2f})"
)

print(
    f"Demand growth: feasible through tested +"
    f"{max_feasible_growth:.0f}%, "
    f"infeasible at tested +{first_infeasible_growth:.0f}%."
)


# 6. Save executive reporting datasets
risk.to_csv(
    TABLES_DIR / "executive_disruption_ranking.csv",
    index=False
)

best_capacity.to_csv(
    TABLES_DIR / "executive_capacity_recommendations.csv",
    index=False
)

demand.to_csv(
    TABLES_DIR / "executive_demand_feasibility.csv",
    index=False
)


# 7. Final validation
assert len(risk) == 4
assert best_investment["lever"] == "SUP_A"
assert np.isclose(max_feasible_growth, 10)
assert np.isclose(first_infeasible_growth, 15)

print("\nSTEP 06: RISK & CAPACITY ANALYSIS PASSED")


Loaded capacity_value.csv: 9 rows

BEST TESTED CAPACITY EXPANSIONS


,lever,extra_weekly_capacity,total_savings_usd,marginal_value_per_unit_week_usd,maximum_12_week_investment_usd
0,SUP_A,405,7883.0,0.229630,7883.0
1,Assembly Plant,116,508.0,0.218391,508.0
2,SUP_B,75,0.0,0.000000,0.0



DISRUPTION COST RANKING


,Scenario,Optimized_Cost_USD,Incremental_Cost_USD,Cost_Change_Pct
2,Demand_Surge,418967.70,14836.50,3.671209
1,Supplier_Disruption,417631.20,13500.00,3.340499
4,Freight_Inflation,413920.45,9789.25,2.422295
3,Manufacturing_Disruption,405741.20,1610.00,0.398385



DEMAND GROWTH FEASIBILITY
Highest tested feasible growth: +10%
First tested infeasible growth: +15%


,Experiment,Demand_Growth_Pct,Status,Optimized_Cost_USD
0,Demand_Growth_Plus_5Pct,5,Optimal,433341.45
1,Demand_Growth_Plus_10Pct,10,Optimal,467764.55
2,Demand_Growth_Plus_15Pct,15,infeasible,NaN
3,Demand_Growth_Plus_20Pct,20,infeasible,NaN



EXECUTIVE FINDINGS
Highest scenario cost increase: Demand_Surge ($14,836.50)
Highest tested capacity savings: SUP_A ($7,883.00)
Demand growth: feasible through tested +10%, infeasible at tested +15%.

STEP 06: RISK & CAPACITY ANALYSIS PASSED


In [13]:
# STEP 07 — Final project deliverable audit

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

dashboard_files = [
    "dashboard_01_network_cost.pdf",
    "dashboard_02_sourcing_manufacturing.pdf",
    "dashboard_03_transportation_distribution.pdf",
    "dashboard_04_risk_scenario_sensitivity.pdf",
]

reporting_files = [
    "baseline_vs_optimized.csv",
    "scenario_comparison.csv",
    "executive_operational_kpis.csv",
    "executive_disruption_ranking.csv",
    "executive_capacity_recommendations.csv",
    "executive_demand_feasibility.csv",
]

audit_records = []

# 1. Check all ten notebooks
for number in range(1, 11):
    matches = list(
        NOTEBOOKS_DIR.glob(f"{number:02d}_*.ipynb")
    )

    audit_records.append({
        "Category": "Notebook",
        "Deliverable": f"Notebook {number:02d}",
        "Found": len(matches) == 1,
        "Location": (
            str(matches[0].relative_to(PROJECT_ROOT))
            if len(matches) == 1 else "MISSING / DUPLICATE"
        )
    })

# 2. Check critical reporting tables
for filename in reporting_files:
    path = TABLES_DIR / filename

    audit_records.append({
        "Category": "Reporting CSV",
        "Deliverable": filename,
        "Found": path.is_file(),
        "Location": (
            str(path.relative_to(PROJECT_ROOT))
            if path.is_file() else "MISSING"
        )
    })

# 3. Check dashboard PDFs
for filename in dashboard_files:
    path = FIGURES_DIR / filename

    valid = (
        path.is_file()
        and path.stat().st_size > 0
        and path.read_bytes()[:4] == b"%PDF"
    )

    audit_records.append({
        "Category": "Dashboard PDF",
        "Deliverable": filename,
        "Found": valid,
        "Location": (
            str(path.relative_to(PROJECT_ROOT))
            if valid else "MISSING / INVALID"
        )
    })

# 4. Check repository documentation
readme = PROJECT_ROOT / "README.md"

audit_records.append({
    "Category": "Documentation",
    "Deliverable": "README.md",
    "Found": readme.is_file() and readme.stat().st_size > 0,
    "Location": "README.md"
})

# 5. Build audit report
audit_df = pd.DataFrame(audit_records)

display(audit_df)

audit_file = TABLES_DIR / "final_deliverable_audit.csv"
audit_df.to_csv(audit_file, index=False)

found_count = int(audit_df["Found"].sum())
total_count = len(audit_df)
missing_count = total_count - found_count

print("\nFINAL DELIVERABLE AUDIT")
print(f"Files validated: {found_count}/{total_count}")
print(f"Missing or invalid: {missing_count}")

if missing_count:
    print("\nITEMS REQUIRING ATTENTION:")
    display(audit_df.loc[~audit_df["Found"]])

    pdf_candidates = list(
        (PROJECT_ROOT / "outputs").rglob("*.pdf")
    )
    print("\nPDFs currently located in outputs/:")
    for p in pdf_candidates:
        print(p.relative_to(PROJECT_ROOT))
else:
    print("All expected deliverables were found.")

print(f"\nAudit saved: {audit_file}")
print("STEP 07: DELIVERABLE AUDIT COMPLETE")


,Category,Deliverable,Found,Location
0,Notebook,Notebook 01,True,notebooks/01_project1_demand_integration.ipynb
1,Notebook,Notebook 02,True,notebooks/02_supply_network_data.ipynb
2,Notebook,Notebook 03,True,notebooks/03_baseline_supply_plan.ipynb
3,Notebook,Notebook 04,True,notebooks/04_optimization_model.ipynb
4,Notebook,Notebook 05,True,notebooks/05_supplier_manufacturing_constraint...
5,Notebook,Notebook 06,True,notebooks/06_distribution_transportation.ipynb
6,Notebook,Notebook 07,True,notebooks/07_scenario_analysis.ipynb
7,Notebook,Notebook 08,True,notebooks/08_sensitivity_analysis.ipynb
8,Notebook,Notebook 09,True,notebooks/09_dashboard_reporting.ipynb
9,Notebook,Notebook 10,True,notebooks/10_executive_summary.ipynb



FINAL DELIVERABLE AUDIT
Files validated: 21/21
Missing or invalid: 0
All expected deliverables were found.

Audit saved: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables/final_deliverable_audit.csv
STEP 07: DELIVERABLE AUDIT COMPLETE


In [14]:
# STEP 08 — GENERATE FINAL EXECUTIVE SUMMARY


from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 8.1 — Initialize report directory
# ------------------------------------------------------------

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

report_path = DOCS_DIR / "executive_summary.md"


# ------------------------------------------------------------
# 8.2 — Load validated reporting datasets
# ------------------------------------------------------------

financial = pd.read_csv(
    TABLES_DIR / "baseline_vs_optimized.csv"
)

operations = pd.read_csv(
    TABLES_DIR / "executive_operational_kpis.csv"
)

risk = pd.read_csv(
    TABLES_DIR / "executive_disruption_ranking.csv"
)

capacity = pd.read_csv(
    TABLES_DIR / "executive_capacity_recommendations.csv"
)

growth = pd.read_csv(
    TABLES_DIR / "executive_demand_feasibility.csv"
)

print("VALIDATED DATASETS LOADED")
print(f"Financial metrics: {len(financial)}")
print(f"Operational KPIs: {len(operations)}")
print(f"Disruption scenarios: {len(risk)}")
print(f"Capacity levers: {len(capacity)}")
print(f"Demand experiments: {len(growth)}")


# ------------------------------------------------------------
# 8.3 — Extract and validate financial performance
# ------------------------------------------------------------

fm = financial.set_index("Metric")["Value"]
op = operations.set_index("Metric")["Value"]

baseline_cost = float(fm["Baseline Cost"])
optimized_cost = float(fm["Optimized Cost"])

network_savings = baseline_cost - optimized_cost
network_savings_pct = network_savings / baseline_cost * 100

freight_savings = float(op["Freight Savings USD"])
freight_reduction_pct = float(op["Freight Reduction (%)"])

supplier_a_share = float(op["Supplier A Share (%)"])
total_production = float(op["Total Production Units"])
customer_fulfillment = float(op["Customer Fulfillment (%)"])

peak_production = float(op["Peak Weekly Production Units"])
weeks_at_capacity = int(op["Weeks at Plant Capacity"])

assert np.isclose(baseline_cost, 414139.70)
assert np.isclose(optimized_cost, 404131.20)
assert np.isclose(network_savings, 10008.50)
assert np.isclose(freight_savings, 8900.00, atol=0.05)
assert np.isclose(customer_fulfillment, 100.0)

print("\nFINANCIAL PERFORMANCE VALIDATED")
print(f"Baseline network cost: ${baseline_cost:,.2f}")
print(f"Optimized network cost: ${optimized_cost:,.2f}")
print(f"Total modeled savings: ${network_savings:,.2f}")
print(f"Cost improvement: {network_savings_pct:.2f}%")


# ------------------------------------------------------------
# 8.4 — Extract disruption scenario results
# ------------------------------------------------------------

risk = risk.sort_values(
    "Incremental_Cost_USD",
    ascending=False
).copy()

highest_risk = risk.iloc[0]

risk_rows = []

for _, row in risk.iterrows():

    scenario_name = str(row["Scenario"]).replace("_", " ")

    scenario_cost = float(row["Optimized_Cost_USD"])
    incremental_cost = float(row["Incremental_Cost_USD"])

    risk_rows.append(
        f"| {scenario_name} | "
        f"${scenario_cost:,.2f} | "
        f"${incremental_cost:,.2f} |"
    )

risk_table = "\n".join(risk_rows)

assert len(risk) == 4

print("\nDISRUPTION RESULTS VALIDATED")
print(
    f"Largest incremental cost: "
    f"{highest_risk['Scenario']} "
    f"(${highest_risk['Incremental_Cost_USD']:,.2f})"
)


# ------------------------------------------------------------
# 8.5 — Extract best tested capacity expansion options
# ------------------------------------------------------------

capacity = capacity.sort_values(
    "total_savings_usd",
    ascending=False
).copy()

best_capacity = capacity.iloc[0]

capacity_rows = []

for _, row in capacity.iterrows():

    lever = str(row["lever"]).replace("_", " ")

    extra_capacity = float(row["extra_weekly_capacity"])
    modeled_savings = float(row["total_savings_usd"])

    capacity_rows.append(
        f"| {lever} | "
        f"{extra_capacity:,.0f} units/week | "
        f"${modeled_savings:,.2f} |"
    )

capacity_table = "\n".join(capacity_rows)

assert best_capacity["lever"] == "SUP_A"
assert np.isclose(
    float(best_capacity["total_savings_usd"]),
    7883.00
)

print("\nCAPACITY INVESTMENT RESULTS VALIDATED")
print(
    f"Best lever: {best_capacity['lever']}"
)
print(
    f"Best tested savings: "
    f"${best_capacity['total_savings_usd']:,.2f}"
)


# ------------------------------------------------------------
# 8.6 — Validate demand growth and feasibility
# ------------------------------------------------------------

growth["Demand_Growth_Pct"] = pd.to_numeric(
    growth["Demand_Growth_Pct"]
)

growth = growth.sort_values("Demand_Growth_Pct")

g5 = growth.loc[
    growth["Demand_Growth_Pct"] == 5
].iloc[0]

g10 = growth.loc[
    growth["Demand_Growth_Pct"] == 10
].iloc[0]

g15 = growth.loc[
    growth["Demand_Growth_Pct"] == 15
].iloc[0]

g20 = growth.loc[
    growth["Demand_Growth_Pct"] == 20
].iloc[0]

cost_5 = float(g5["Optimized_Cost_USD"])
cost_10 = float(g10["Optimized_Cost_USD"])

increase_5 = cost_5 - optimized_cost
increase_10 = cost_10 - optimized_cost

increase_5_pct = increase_5 / optimized_cost * 100
increase_10_pct = increase_10 / optimized_cost * 100

assert np.isclose(cost_5, 433341.45)
assert np.isclose(cost_10, 467764.55)

assert str(g5["Status"]).lower() == "optimal"
assert str(g10["Status"]).lower() == "optimal"
assert str(g15["Status"]).lower() == "infeasible"
assert str(g20["Status"]).lower() == "infeasible"

print("\nDEMAND SENSITIVITY VALIDATED")
print(f"+5% demand cost: ${cost_5:,.2f}")
print(f"+10% demand cost: ${cost_10:,.2f}")
print("+15% and +20% demand: Infeasible")


# ------------------------------------------------------------
# 8.7 — Generate the executive report
# ------------------------------------------------------------

report = f"""# End-to-End Hardware Supply Network Optimizer

## V1 Executive Summary & Final Results

**Report Date:** {date.today().strftime("%B %d, %Y")}  
**Planning Horizon:** 12 Weeks  
**Technology Stack:** Python, pandas, Pyomo, HiGHS, SQL, Tableau  
**Project Type:** Supply Chain Optimization & Decision Analytics

---

## 1. Executive Summary

This project developed an integrated optimization framework
for a hypothetical hardware supply network connecting supplier
sourcing, component inventory, manufacturing, transportation,
distribution centers, and customer fulfillment.

A rule-based baseline supply plan was compared against
a mathematical optimization model using Pyomo and HiGHS.

The optimized network reduced modeled total costs from
**${baseline_cost:,.2f}** to **${optimized_cost:,.2f}**,
generating **${network_savings:,.2f} in savings**,
equivalent to a **{network_savings_pct:.2f}% reduction**.

Both plans achieved **100% modeled customer fulfillment**.

The greatest positive source of cost improvement was
transportation optimization, generating
**${freight_savings:,.2f} in freight savings**.

---

## 2. Business Problem

Hardware supply chains must coordinate multiple operational
decisions while operating under capacity, lead-time, and
inventory constraints.

These decisions include:

- Selecting suppliers and sourcing quantities.
- Planning component procurement and replenishment.
- Scheduling production against manufacturing capacity.
- Choosing transportation modes and distribution routes.
- Positioning finished-goods inventory across distribution centers.
- Meeting regional customer demand.
- Responding to supplier and manufacturing disruptions.

The objective was to minimize modeled total network cost
without sacrificing customer fulfillment.

---

## 3. Optimization Methodology

The project followed an integrated analytical workflow:

1. Import demand forecasts from Project 1.
2. Define hypothetical network operating assumptions.
3. Construct a rule-based baseline supply plan.
4. Formulate a constrained mathematical optimization model.
5. Solve the optimization problem using Pyomo and HiGHS.
6. Analyze supplier sourcing and manufacturing utilization.
7. Evaluate transportation and distribution decisions.
8. Reoptimize under hypothetical disruption scenarios.
9. Test capacity expansion and demand sensitivity.
10. Generate Tableau dashboards and executive reporting outputs.

The optimization model included supplier capacity,
supplier lead times, bill-of-materials requirements,
component inventory balance, manufacturing capacity,
transportation lane capacity, distribution center storage,
customer fulfillment, and terminal inventory constraints.

---

## 4. Financial Performance

| Performance Metric | Result |
|---|---:|
| Baseline Network Cost | ${baseline_cost:,.2f} |
| Optimized Network Cost | ${optimized_cost:,.2f} |
| Total Modeled Savings | ${network_savings:,.2f} |
| Network Cost Reduction | {network_savings_pct:.2f}% |
| Freight Savings | ${freight_savings:,.2f} |
| Freight Cost Reduction | {freight_reduction_pct:.2f}% |
| Customer Fulfillment | {customer_fulfillment:.1f}% |

### Interpretation

Transportation optimization accounted for the largest
positive savings contribution.

The model improved coordination between production,
supplier procurement, distribution inventory, and freight
mode selection while preserving modeled service levels.

---

## 5. Sourcing and Manufacturing Performance

### Supplier Allocation

Supplier A accounted for **{supplier_a_share:.1f}%**
of modeled component purchasing.

This concentration reflects the optimizer's preference
for lower-cost sourcing but introduces a potential
supplier-dependence risk.

### Manufacturing Operations

| Manufacturing KPI | Result |
|---|---:|
| Total Finished-Goods Production | {total_production:,.0f} units |
| Peak Weekly Production | {peak_production:,.0f} units |
| Weeks at Plant Capacity | {weeks_at_capacity} |

Manufacturing constraints were respected throughout
the optimized planning horizon.

---

## 6. Transportation and Distribution Performance

The optimized network reduced modeled freight expense
by **${freight_savings:,.2f}**,
equivalent to a **{freight_reduction_pct:.2f}% reduction**.

The optimization achieved these savings through
changes in standard versus expedited freight allocation,
transportation lane utilization, and timing of shipments.

Both distribution centers supported **100% customer
fulfillment** under the reference planning assumptions.

---

## 7. Disruption Scenario Analysis

The network was reoptimized under four hypothetical
operational disruptions.

| Disruption Scenario | Total Network Cost | Incremental Cost |
|---|---:|---:|
{risk_table}

### Interpretation

The largest incremental cost occurred under
**{str(highest_risk["Scenario"]).replace("_", " ")}**,
increasing modeled network expense by
**${float(highest_risk["Incremental_Cost_USD"]):,.2f}**.

The demand-surge scenario included additional customer
volume, so its increase is not solely a disruption penalty.

These results demonstrate the financial consequences
of operational uncertainty under the modeled constraints.

---

## 8. Capacity Expansion Economics

The sensitivity analysis evaluated alternative
supplier and manufacturing capacity expansions.

| Capacity Lever | Extra Weekly Capacity | Gross Modeled Savings |
|---|---:|---:|
{capacity_table}

### Highest-Value Opportunity

The best tested capacity expansion was
**{str(best_capacity["lever"]).replace("_", " ")}**,
with **{float(best_capacity["extra_weekly_capacity"]):,.0f}
additional units of weekly capacity**.

This experiment generated
**${float(best_capacity["total_savings_usd"]):,.2f}
in gross modeled operating savings**.

Actual expansion investments, qualification costs,
and ongoing expenses must be evaluated before
treating these savings as a net investment benefit.

---

## 9. Demand Growth Sensitivity

The model was tested against additional
customer demand.

| Demand Growth | Optimized Network Cost | Status |
|---|---:|---|
| +5% | ${cost_5:,.2f} | Optimal |
| +10% | ${cost_10:,.2f} | Optimal |
| +15% | Not available | Infeasible |
| +20% | Not available | Infeasible |

At +5% demand growth, modeled network cost increased
by **${increase_5:,.2f} ({increase_5_pct:.2f}%)**.

At +10% demand growth, modeled network cost increased
by **${increase_10:,.2f} ({increase_10_pct:.2f}%)**.

The tested +15% and +20% scenarios were infeasible
under the current network constraints.

These experiments identify potential scalability
limitations, although they do not establish the
exact maximum feasible demand increase.

---

## 10. Executive Recommendations

### Recommendation 1 — Prioritize High-Value Capacity

Evaluate additional Supplier A capacity against
its estimated gross modeled operating savings
and the actual cost of expansion.

### Recommendation 2 — Manage Supplier Concentration

Investigate dual sourcing, contingency inventory,
and supplier diversification for critical components.

### Recommendation 3 — Improve Transportation Planning

Preserve opportunities to reduce expedited freight
through coordinated production and shipment scheduling.

### Recommendation 4 — Investigate Capacity Bottlenecks

Identify the binding constraints responsible
for demand-growth infeasibility.

Evaluate supplier capacity, production flexibility,
transportation capacity, and inventory positioning
before expanding demand commitments.

### Recommendation 5 — Build Resilience into Planning

Use disruption scenarios to evaluate the trade-offs
between operating efficiency, flexibility, and
risk exposure.

---

## 11. Executive Tableau Dashboards

Four dashboards summarize the project's results.

1. **Network Cost & Optimization Performance**
2. **Sourcing & Manufacturing Intelligence**
3. **Transportation & Distribution Intelligence**
4. **Risk, Scenario & Sensitivity Intelligence**

PDF exports are stored in:

`outputs/figures/`

The demand-growth comparison in Dashboard 4
should reflect the validated **+10% experiment**.

---

## 12. Modeling Limitations

This project uses hypothetical operating assumptions.

Important limitations include:

- Synthetic supplier, manufacturing, and distribution parameters.
- Deterministic costs, capacities, and lead times.
- No probability-weighted disruption forecasts.
- Limited demand-growth experiments.
- Simplified inventory and transportation assumptions.
- Capacity investment expenses not fully modeled.
- No guarantee that modeled savings translate
  into realized business savings.
- Results depend on the accuracy of input demand assumptions.

These limitations provide opportunities for
future model refinement and validation.

---

## 13. Conclusion

The End-to-End Hardware Supply Network Optimizer
demonstrates how mathematical optimization can
integrate sourcing, manufacturing, inventory,
transportation, and distribution decisions.

The model delivered a **{network_savings_pct:.2f}%
reduction in modeled network costs** while
maintaining **{customer_fulfillment:.1f}% customer
fulfillment** under reference conditions.

Disruption and sensitivity experiments also identified
sourcing concentration, cost exposure, and
capacity limitations that should inform
future network planning decisions.

The completed project provides a foundation
for expanded stochastic modeling, deeper
supplier-risk analysis, and more advanced
hardware supply chain decision support.

---

*End of Project 2 — V1 Executive Summary*
"""


# ------------------------------------------------------------
# 8.8 — Save Markdown executive summary
# ------------------------------------------------------------

report_path.write_text(
    report,
    encoding="utf-8"
)

assert report_path.exists()
assert report_path.stat().st_size > 1000

print("\n" + "=" * 65)
print("FINAL EXECUTIVE REPORT GENERATED")
print("=" * 65)

print(f"Report: {report_path}")
print(f"Characters: {len(report):,}")
print(f"File size: {report_path.stat().st_size:,} bytes")


# ------------------------------------------------------------
# 8.9 — Final financial reconciliation
# ------------------------------------------------------------

print("\nFINAL FINANCIAL RECONCILIATION")

print(f"Baseline Cost:      ${baseline_cost:,.2f}")
print(f"Optimized Cost:     ${optimized_cost:,.2f}")
print(f"Network Savings:    ${network_savings:,.2f}")
print(f"Improvement:        {network_savings_pct:.2f}%")
print(f"Freight Savings:    ${freight_savings:,.2f}")

print("\nDEMAND SENSITIVITY RECONCILIATION")

print(f"+5% Demand Increase:  ${increase_5:,.2f}")
print(f"+10% Demand Increase: ${increase_10:,.2f}")

print("\nSTEP 08: EXECUTIVE REPORT GENERATED")

if "audit_df" in globals() and not audit_df["Found"].all():
    print(
        "\nWARNING: Some deliverables remain incomplete. "
        "Review Step 07 before publication."
    )
else:
    print(
        "\nFinal report saved successfully. "
        "Review dashboard content and documentation before GitHub V1 publication."
    )

VALIDATED DATASETS LOADED
Financial metrics: 4
Operational KPIs: 11
Disruption scenarios: 4
Capacity levers: 3
Demand experiments: 4

FINANCIAL PERFORMANCE VALIDATED
Baseline network cost: $414,139.70
Optimized network cost: $404,131.20
Total modeled savings: $10,008.50
Cost improvement: 2.42%

DISRUPTION RESULTS VALIDATED
Largest incremental cost: Demand_Surge ($14,836.50)

CAPACITY INVESTMENT RESULTS VALIDATED
Best lever: SUP_A
Best tested savings: $7,883.00

DEMAND SENSITIVITY VALIDATED
+5% demand cost: $433,341.45
+10% demand cost: $467,764.55
+15% and +20% demand: Infeasible

FINAL EXECUTIVE REPORT GENERATED
Report: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/docs/executive_summary.md
Characters: 9,198
File size: 9,210 bytes

FINAL FINANCIAL RECONCILIATION
Baseline Cost:      $414,139.70
Optimized Cost:     $404,131.20
Network Savings:    $10,008.50
Improvement:        2.42%
Freight Savings:    $8,900.00

DEMAND SENSITIVITY RECONCILIATION
+5